# RQ3: anchor

**CHỈ CHẠY TRÊN COLAB/KAGGLE HOSTED RUNTIME. CẤM MÁY CÁ NHÂN / LOCAL RUNTIME.**

Đọc [hướng dẫn bàn giao](../README.md), cài requirements trên cloud trước.


In [ ]:
import os
import sys
from pathlib import Path

# Cấm local runtime, kể cả máy cá nhân có CUDA.
if not ((os.environ.get("COLAB_RELEASE_TAG") and Path("/content").is_dir()) or
        (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") and Path("/kaggle/working").is_dir())):
    raise RuntimeError("RULES.md §8: chỉ Colab/Kaggle hosted runtime; KHÔNG chạy trên máy cá nhân.")

# Clone/mount repo trước, đặt cwd vào repo; không tự tải dataset.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "requirements_learned.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Chuyển working directory tới bản clone cvprj trên cloud trước khi chạy.")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "notebooks" / "shared"))
from cloud_runtime import job_log

# Colab: mount Google Drive trước. Kaggle: Save Version có outputs để giữ artifact.
OUTPUT_ROOT = Path(os.environ.get("UAV_OUTPUT_ROOT", "/kaggle/working/uav_runs" if
                   os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "/content/drive/MyDrive/uav_runs"))
if not OUTPUT_ROOT.parent.is_dir():
    raise RuntimeError("Mount kho lưu bền vững hoặc đặt UAV_OUTPUT_ROOT trước khi chạy.")
DEVICE = "cpu"  # CPU CLOUD; dùng "cuda" nhất quán cho mọi job của cùng benchmark nếu cần.


## Cấu hình công việc và đầu vào
Chỉ điền đường dẫn artifact cụ thể, không tự chọn run mới nhất.

In [ ]:
SCAN_PATHS = []  # ba scan.json, mỗi matcher đúng một file; cùng dữ liệu/config/device
POLICY = 'anchor'
RQ2_PATH = ""  # result.json từ 05_rq2


## Chạy và lưu log
Mỗi lần chạy có thư mục UTC riêng; không ghi đè artifact paper cũ.

In [ ]:
if len(SCAN_PATHS) != 3:
    raise ValueError("Cần ba đường dẫn scan.json được bàn giao")
import research_jobs as jobs
jobs.configure(ROOT, DEVICE)
if not RQ2_PATH:
    raise ValueError("Điền RQ2_PATH để dùng cùng ngưỡng")
with job_log(OUTPUT_ROOT, "05_recovery_" + POLICY) as run_root:
    result = jobs.recovery_job(SCAN_PATHS, RQ2_PATH, POLICY, run_root)
print(result / "result.json")
